# Model evaluation

Kendra Wyant  
March 5, 2026

### Set Up Environment

In [ ]:
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(source("https://github.com/jjcurtin/lab_support/blob/main/format_path.R?raw=true"))
suppressPackageStartupMessages(library(tidyposterior))

path_models <- format_path(str_c("risk2/models/combined"))


In [ ]:
test_metrics_full <- read_csv(here::here(path_models, 
                                    "test_auroc_6_x_5_1_x_5_day_v10_nested_full.csv"), 
                              show_col_types = FALSE) |> 
  select(split_num = outer_split_num, "full" = roc_auc) |> 
  arrange(split_num)

test_metrics_ablate_both <- read_csv(here::here(path_models, 
                                    "test_auroc_6_x_5_1_x_5_day_v10_nested_ablate_both.csv"), 
                              show_col_types = FALSE) |> 
  select(split_num = outer_split_num, "ablated daily survey and geolocation" = roc_auc) |> 
  arrange(split_num)

test_metrics_ablate_ema <- read_csv(here::here(path_models, 
                                    "test_auroc_6_x_5_1_x_5_day_v10_nested_ablate_ema.csv"), 
                              show_col_types = FALSE) |> 
  select(split_num = outer_split_num, "ablated daily survey" = roc_auc) |> 
  arrange(split_num)

test_metrics_ablate_gps <- read_csv(here::here(path_models, 
                                    "test_auroc_6_x_5_1_x_5_day_v10_nested_ablate_gps.csv"), 
                              show_col_types = FALSE) |> 
  select(split_num = outer_split_num, "ablated geolocation" = roc_auc) |> 
  arrange(split_num)

test_metrics <- test_metrics_full |> 
  left_join(test_metrics_ablate_both, by = c("split_num")) |> 
  left_join(test_metrics_ablate_ema, by = c("split_num")) |> 
  left_join(test_metrics_ablate_gps, by = c("split_num")) |> 
  mutate(fold_num = rep(1:5, 6),
         repeat_num = c(rep(1, 5), rep(2, 5), rep(3, 5), 
                        rep(4, 5), rep(5, 5), rep(6, 5))) |> 
  select(-split_num) |> 
  glimpse()


Rows: 30
Columns: 6
$ full                                   <dbl> 0.9230050, 0.9213850, 0.9078228…
$ `ablated daily survey and geolocation` <dbl> 0.6081222, 0.3882430, 0.4349535…
$ `ablated daily survey`                 <dbl> 0.6096729, 0.5275498, 0.5979393…
$ `ablated geolocation`                  <dbl> 0.9169196, 0.9170817, 0.9235670…
$ fold_num                               <int> 1, 2, 3, 4, 5, 1, 2, 3, 4, 5, 1…
$ repeat_num                             <dbl> 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3…

#### Model evaluation

In [ ]:

# Repeated CV (id = repeat, id2 = fold within repeat)
# with a common variance:  statistic ~ model + (model | id2/id)
set.seed(101)
pp <- test_metrics |> 
  rename(id = fold_num,
         id2 = repeat_num) |> 
  perf_mod(formula = statistic ~ model + (1 | id2/id),
         transform = tidyposterior::logit_trans,  # for skewed & bounded AUC
         iter = 6000, chains = 4, adapt_delta = .99, # increased iteration from 2000 to fix divergence issues
         family = gaussian, 
)  



SAMPLING FOR MODEL 'continuous' NOW (CHAIN 1).
Chain 1: 
Chain 1: Gradient evaluation took 8.6e-05 seconds
Chain 1: 1000 transitions using 10 leapfrog steps per transition would take 0.86 seconds.
Chain 1: Adjust your expectations accordingly!
Chain 1: 
Chain 1: 
Chain 1: Iteration:    1 / 6000 [  0%]  (Warmup)
Chain 1: Iteration:  600 / 6000 [ 10%]  (Warmup)
Chain 1: Iteration: 1200 / 6000 [ 20%]  (Warmup)
Chain 1: Iteration: 1800 / 6000 [ 30%]  (Warmup)
Chain 1: Iteration: 2400 / 6000 [ 40%]  (Warmup)
Chain 1: Iteration: 3000 / 6000 [ 50%]  (Warmup)
Chain 1: Iteration: 3001 / 6000 [ 50%]  (Sampling)
Chain 1: Iteration: 3600 / 6000 [ 60%]  (Sampling)
Chain 1: Iteration: 4200 / 6000 [ 70%]  (Sampling)
Chain 1: Iteration: 4800 / 6000 [ 80%]  (Sampling)
Chain 1: Iteration: 5400 / 6000 [ 90%]  (Sampling)
Chain 1: Iteration: 6000 / 6000 [100%]  (Sampling)
Chain 1: 
Chain 1:  Elapsed Time: 3.529 seconds (Warm-up)
Chain 1:                4.077 seconds (Sampling)
Chain 1:                7.60

In [ ]:
pp_tidy <- pp |> 
  tidy(seed = 123) 

q = c(.025, .5, .975)
pp_perf_tibble <- pp_tidy |> 
  group_by(model) |> 
  summarize(pp_median = quantile(posterior, probs = q[2]),
            pp_lower = quantile(posterior, probs = q[1]), 
            pp_upper = quantile(posterior, probs = q[3])) |> 
  mutate(model = factor(model, levels = c("full", "ablated daily survey", "ablated geolocation", "ablated daily survey and geolocation"))) |> 
  arrange(model) 

pp_perf_tibble |> 
  write_csv(here::here(path_models, "pp_perf_tibble_v10.csv"))

pp_tidy |> 
  write_csv(here::here(path_models, "posteriors_v10.csv"))

pp_perf_tibble


# A tibble: 4 × 4
  model                                pp_median pp_lower pp_upper
  <fct>                                    <dbl>    <dbl>    <dbl>
1 full                                     0.914    0.904    0.923
2 ablated daily survey                     0.620    0.591    0.648
3 ablated geolocation                      0.916    0.906    0.925
4 ablated daily survey and geolocation     0.551    0.519    0.581

### Model Comparison

In [ ]:
ci_ablate <- pp |>
  contrast_models(list("full", "full", "full"),
                  list("ablated daily survey", "ablated geolocation", "ablated daily survey and geolocation")) |>
  summary(size = 0) |> 
  mutate(contrast = factor(contrast,
                           levels = c("full vs ablated daily survey",
                                      "full vs ablated geolocation",
                                      "full vs ablated daily survey and geolocation"),
                           labels = c("full vs. ablated daily survey",
                                      "full vs. ablated geolocation",
                                      "full vs. ablated daily survey and geolocation"))) |> 
  arrange(contrast)

ci_median <- pp |>
  contrast_models(list("full", "full", "full"),
                  list("ablated daily survey", "ablated geolocation", "ablated daily survey and geolocation")) |>
  group_by(contrast) |>
  summarize(median = quantile(difference, .5))


ci_ablate <- ci_ablate |>
  left_join(ci_median, by = c("contrast")) 

ci_ablate |>
  write_csv(here::here(path_models, "contrast_ablate_v10.csv"))

ci_ablate


# A tibble: 3 × 10
  contrast      probability     mean   lower   upper  size pract_neg pract_equiv
  <chr>               <dbl>    <dbl>   <dbl>   <dbl> <dbl>     <dbl>       <dbl>
1 full vs. abl…       1      0.294    0.271  0.317       0        NA          NA
2 full vs. abl…       0.350 -0.00220 -0.0115 0.00718     0        NA          NA
3 full vs. abl…       1      0.363    0.339  0.388       0        NA          NA
# ℹ 2 more variables: pract_pos <dbl>, median <dbl>